In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index, 
    llm_client=openai_client,
    instructions=instructions,
)


In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, first install it from [https://ollama.com/download](https://ollama.com/download) for your operating system:\n\n- **macOS**: download and install the `.pkg`\n- **Windows**: download and install the `.msi`\n- **Linux**: run:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nAfter installing, open a terminal and run:\n\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo test that the local server is running, you can use:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n\n```json\n{"models": [...]}\n```\n\nIf you’re using Python, you can also install the client with:\n\n```bash\npip install ollama\n```\n\nand use it in code with `ollama.chat(...)`.'

In [5]:
assistant.rag("How do I run Olama locally?")

'I don’t see any information in the context about running **Ollama** locally.\n\nThe closest related note is that you can run the **course locally** instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and other needed tools, and that you should keep your environment reproducible.'

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Usually, yes — if the course is still open and you meet any prerequisites.\n\nA quick way to check:\n1. See whether enrollment is still available.\n2. Check for prerequisites or required prior knowledge.\n3. Contact the course organizer or instructor to ask if late registration is possible.\n\nIf you want, I can help you write a short message asking to join the course.'

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join new discovery"}', call_id='call_SjAPfhLmYQshrWcnBkguycq4', name='search', type='function_call', id='fc_0168b44a27607850006a410c788b508193bc102e6995a58b79', namespace=None, status='completed')]

In [10]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [11]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning anytime.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [14]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(925, 33)

In [15]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [17]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [18]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

In [20]:
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course new student enrollment can I join discovered the course"}
function_call: search {"query":"course enrollment late join discovered the course FAQ"}
function_call: search {"query":"self-paced join course anytime enrollment FAQ"}


In [23]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join the course enrollment discovered course can I join"}
iteration #2...
function_call: search {"query":"certificate submit project accepting submissions still join course live cohort self-paced"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your capstone/project while submissions are still open. Also, certificates are only available for the live cohort, not self-paced participation.

If you'd like, I can also explain how to start the course or what the certificate requirements are. Any other areas you want to explore?


In [24]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [25]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama locally run install setup local model course FAQ"}
function_call: search {"query":"how to run Ollama locally FAQ"}
function_call: search {"query":"Ollama local run model command ollama run FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s working, you should get a response listing models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   response = ollama.chat(
       mo

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a response listing models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Then:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a “connection refused” error, restart the server with:\n```bash\nollama serve\n```\nor in a no

In [26]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"course FAQ enrollment late join registration discovered course"}
function_call: search {"query":"can I still join the course after it started FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still open. You can also start learning and working through the materials right away.

If you’d like, I can also help you figure out the best way to catch up quickly. Are there other areas you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, the key thing is to submit your project while submissions are still open. You can also start learning and working through the materials right away.\n\nIf you’d like, I can also help you figure out the best way to catch up quickly. Are there other areas you want to explore?'

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course enroll discovered course can I join FAQ"}
iteration #2...
function_call: search {"query":"certificate project still accepting submissions peer review live cohort join course registration accepted no need email FAQ"}
iteration #3...
ASSISTANT:
Yes, you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available when you finish with a live cohort, not in self-paced mode.

If you’d like, I can also explain how to start the course now and what the weekly workflow looks like. Any other areas you want to explore?


'Yes, you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available when you finish with a live cohort, not in self-paced mode.\n\nIf you’d like, I can also explain how to start the course now and what the weekly workflow looks like. Any other areas you want to explore?'

In [28]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening definition"}
function_call: search {"query":"queen's gambit chess opening"}
function_call: search {"query":"what is the queen gambit"}
iteration #2...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**. It starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a “gambit” to try to gain control of the center.  
Black can accept it or decline it.

Common lines include:
- **Queen’s Gambit Accepted**: Black takes the pawn on c4
- **Queen’s Gambit Declined**: Black does not take it and defends the center

It’s one of the oldest and most respected openings in chess.

If you meant the **TV show** *The Queen’s Gambit*, I can explain that too. Any other areas you want to explore?


'The **Queen’s Gambit** is a classic **chess opening**. It starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a “gambit” to try to gain control of the center.  \nBlack can accept it or decline it.\n\nCommon lines include:\n- **Queen’s Gambit Accepted**: Black takes the pawn on c4\n- **Queen’s Gambit Declined**: Black does not take it and defends the center\n\nIt’s one of the oldest and most respected openings in chess.\n\nIf you meant the **TV show** *The Queen’s Gambit*, I can explain that too. Any other areas you want to explore?'

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so I’m not able to answer it from the course materials.

If you meant something course-related, feel free to rephrase it. Are there other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” so I’m not able to answer it from the course materials.\n\nIf you meant something course-related, feel free to rephrase it. Are there other areas you want to explore?'